In [4]:
"""
Feature Engineering — Home Credit Default Risk
=================================================
Chuẩn bị input cho kiến trúc Hierarchical Multimodal MoE (cùng tinh thần với
project ADNI gốc), mapping mỗi bảng dữ liệu Home Credit thành một "modality":

    application            -> CORE, luôn có (tương ứng vai trò MRI trong bản gốc)
    bureau (+ bureau_balance) -> optional modality
    previous_application   -> optional modality
    POS_CASH_balance        -> optional modality
    credit_card_balance     -> optional modality
    installments_payments   -> optional modality

Mỗi optional modality trả về:
    X_<name>          : ma trận feature đã fillna(0), shape (n_customers, n_features)
    mask_<name>_feat  : mask từng feature (1 = có dữ liệu gốc, 0 = bị fill do thiếu)
    modality_mask cột tương ứng trong level2_modality_mask
                       (1 = khách hàng có ít nhất 1 bản ghi ở bảng này)
    cols               : danh sách tên cột thực tế (lưu trong metadata JSON)

Cách tiếp cận: aggregation phổ biến nhất cho bài toán này trên Kaggle (tương tự
tinh thần kernel "LightGBM with Simple Features" của Aguiar và "Introduction to
Manual Feature Engineering" của Will Koehrsen), nhưng CHỌN LỌC số lượng feature
mỗi bảng (thay vì vài trăm cột) để phù hợp với kiến trúc MoE — mỗi bảng chỉ cần
một vector đặc trưng vừa đủ diễn giải cho 1 "expert".

Yêu cầu: tải 7 file CSV từ cuộc thi "Home Credit Default Risk" trên Kaggle và
đặt vào DATA_DIR bên dưới (chỉnh lại đường dẫn cho phù hợp máy bạn):
    application_train.csv, bureau.csv, bureau_balance.csv,
    previous_application.csv, POS_CASH_balance.csv,
    credit_card_balance.csv, installments_payments.csv

Chạy: python home_credit_feature_engineering.py
Output: outputs/home_credit_features.npz + outputs/home_credit_feature_metadata.json
"""

import os
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG — chỉnh DATA_DIR cho đúng máy bạn
# ============================================================
import os

# Cách dùng dấu r"..." là an toàn nhất cho đường dẫn Windows
DATA_DIR = r"D:\2026\home_credit" 

PATHS = {
    "application_train": os.path.join(DATA_DIR, "application_train.csv"),
    "application_test": os.path.join(DATA_DIR, "application_test.csv"), # Nên thêm file test
    "bureau": os.path.join(DATA_DIR, "bureau.csv"),
    "bureau_balance": os.path.join(DATA_DIR, "bureau_balance.csv"),
    "previous_application": os.path.join(DATA_DIR, "previous_application.csv"),
    "pos_cash": os.path.join(DATA_DIR, "POS_CASH_balance.csv"),
    "credit_card": os.path.join(DATA_DIR, "credit_card_balance.csv"),
    "installments": os.path.join(DATA_DIR, "installments_payments.csv"),
    "output_dir": "./outputs",
}

# Kiểm tra xem thư mục output đã tồn tại chưa, nếu chưa thì tạo mới
if not os.path.exists(PATHS["output_dir"]):
    os.makedirs(PATHS["output_dir"])
    print(f"Đã tạo thư mục: {PATHS['output_dir']}")

ID_COL = "SK_ID_CURR"
LABEL_COL = "TARGET"

# Các cột nhị phân trong application: label-encode (0/1) thay vì one-hot
BINARY_CATEGORICAL_COLS = ["NAME_CONTRACT_TYPE", "CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY"]


# ============================================================
# HELPERS
# ============================================================

def safe_ratio(numerator, denominator):
    """Chia an toàn: mẫu số = 0 -> NaN thay vì inf (NaN sẽ được xử lý bởi mask/fillna sau)."""
    denom = denominator.replace(0, np.nan)
    return numerator / denom


def aggregate_table(df, group_col, agg_dict, prefix):
    """Group theo group_col, áp agg_dict, flatten tên cột thành PREFIX_COL_STAT."""
    agg_dict = {k: v for k, v in agg_dict.items() if k in df.columns}
    grouped = df.groupby(group_col).agg(agg_dict)
    grouped.columns = [f"{prefix}_{'_'.join(c)}".upper() for c in grouped.columns]
    return grouped.reset_index()


def make_modality_matrix(base_ids, agg_df, id_col=ID_COL):
    """
    Merge agg_df (có thể thiếu 1 số SK_ID_CURR do khách không có bản ghi ở bảng này)
    vào toàn bộ base_ids. Trả về X (fillna 0), feature_mask, modality_mask, cols —
    cùng cấu trúc với make_matrix_and_mask() trong project ADNI gốc.
    """
    full = pd.DataFrame({id_col: base_ids}).merge(agg_df, on=id_col, how="left")
    cols = [c for c in full.columns if c != id_col]

    feature_mask = (~full[cols].isna()).astype(np.float32).values
    modality_mask = (feature_mask.sum(axis=1, keepdims=True) > 0).astype(np.float32)
    X = full[cols].fillna(0).values.astype(np.float32)
    return X, feature_mask, modality_mask, cols


# ============================================================
# 1. CORE — application (luôn có, tương đương MRI trong bản gốc)
# ============================================================

def build_application_features(path):
    df = pd.read_csv(path)
    has_target = LABEL_COL in df.columns

    # Anomaly kinh điển của bảng này: DAYS_EMPLOYED = 365243 nghĩa là "không có việc làm"
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

    # Các tỷ lệ được dùng phổ biến nhất trong các kernel Home Credit
    df["CREDIT_INCOME_RATIO"] = safe_ratio(df["AMT_CREDIT"], df["AMT_INCOME_TOTAL"])
    df["ANNUITY_INCOME_RATIO"] = safe_ratio(df["AMT_ANNUITY"], df["AMT_INCOME_TOTAL"])
    df["CREDIT_TERM"] = safe_ratio(df["AMT_ANNUITY"], df["AMT_CREDIT"])
    df["DAYS_EMPLOYED_BIRTH_RATIO"] = safe_ratio(df["DAYS_EMPLOYED"], df["DAYS_BIRTH"])
    df["INCOME_PER_PERSON"] = safe_ratio(df["AMT_INCOME_TOTAL"], df["CNT_FAM_MEMBERS"])
    df["CREDIT_GOODS_RATIO"] = safe_ratio(df["AMT_CREDIT"], df["AMT_GOODS_PRICE"])

    # Label-encode các cột nhị phân
    for col in BINARY_CATEGORICAL_COLS:
        if col in df.columns:
            df[col] = df[col].astype("category").cat.codes.replace(-1, np.nan)

    # One-hot các cột phân loại còn lại (đa giá trị, VD: ORGANIZATION_TYPE, NAME_EDUCATION_TYPE...)
    obj_cols = [c for c in df.select_dtypes(include="object").columns if c not in BINARY_CATEGORICAL_COLS]
    df = pd.get_dummies(df, columns=obj_cols, dummy_na=True)

    y = df[LABEL_COL].values if has_target else None
    feature_cols = [c for c in df.columns if c not in [ID_COL, LABEL_COL]]

    return df[[ID_COL] + feature_cols], feature_cols, y


# ============================================================
# 2. BUREAU + BUREAU_BALANCE
# ============================================================

def build_bureau_features(bureau_path, balance_path):
    bureau = pd.read_csv(bureau_path)

    if os.path.exists(balance_path):
        balance = pd.read_csv(balance_path)
        status_dummies = pd.get_dummies(balance["STATUS"], prefix="STATUS")
        balance_enc = pd.concat([balance[["SK_ID_BUREAU", "MONTHS_BALANCE"]], status_dummies], axis=1)

        agg_dict = {"MONTHS_BALANCE": ["min", "max", "count"]}
        agg_dict.update({c: "mean" for c in status_dummies.columns})
        balance_agg = balance_enc.groupby("SK_ID_BUREAU").agg(agg_dict)
        balance_agg.columns = [f"BB_{'_'.join(c)}".upper() for c in balance_agg.columns]
        balance_agg = balance_agg.reset_index()

        bureau = bureau.merge(balance_agg, on="SK_ID_BUREAU", how="left")

    bureau["CREDIT_ACTIVE_FLAG"] = (bureau["CREDIT_ACTIVE"] == "Active").astype(int)

    agg_dict = {
        "SK_ID_BUREAU": ["count"],                      # số khoản vay đã từng có ở TCTD khác
        "CREDIT_ACTIVE_FLAG": ["sum", "mean"],           # số / tỷ lệ khoản đang active
        "DAYS_CREDIT": ["min", "max", "mean"],           # độ "cũ" của các khoản vay
        "CREDIT_DAY_OVERDUE": ["max", "mean"],           # mức độ quá hạn
        "AMT_CREDIT_SUM": ["sum", "mean", "max"],
        "AMT_CREDIT_SUM_DEBT": ["sum", "mean"],
        "AMT_CREDIT_SUM_OVERDUE": ["sum", "mean"],
        "AMT_CREDIT_SUM_LIMIT": ["sum", "mean"],
        "CNT_CREDIT_PROLONG": ["sum"],
        "CREDIT_TYPE": ["nunique"],
    }
    return aggregate_table(bureau, ID_COL, agg_dict, prefix="BUREAU")


# ============================================================
# 3. PREVIOUS_APPLICATION
# ============================================================

def build_previous_features(path):
    prev = pd.read_csv(path)

    # 365243 cũng là mã "không áp dụng" trong các cột DAYS_* của bảng này
    for col in ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION",
                "DAYS_LAST_DUE", "DAYS_TERMINATION"]:
        if col in prev.columns:
            prev[col] = prev[col].replace(365243, np.nan)

    prev["APP_CREDIT_RATIO"] = safe_ratio(prev["AMT_APPLICATION"], prev["AMT_CREDIT"])
    prev["APPROVED_FLAG"] = (prev["NAME_CONTRACT_STATUS"] == "Approved").astype(int)
    prev["REFUSED_FLAG"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(int)

    agg_dict = {
        "SK_ID_PREV": ["count"],                 # số lần từng apply vay ở Home Credit
        "AMT_ANNUITY": ["mean", "max"],
        "AMT_APPLICATION": ["mean", "max"],
        "AMT_CREDIT": ["mean", "max"],
        "APP_CREDIT_RATIO": ["mean"],
        "APPROVED_FLAG": ["sum", "mean"],        # tỷ lệ được duyệt trước đây
        "REFUSED_FLAG": ["sum", "mean"],         # tỷ lệ bị từ chối trước đây
        "CNT_PAYMENT": ["mean"],
        "DAYS_DECISION": ["min", "max", "mean"],
    }
    return aggregate_table(prev, ID_COL, agg_dict, prefix="PREV")


# ============================================================
# 4. POS_CASH_BALANCE
# ============================================================

def build_pos_features(path):
    pos = pd.read_csv(path)
    pos["LATE_PAYMENT_FLAG"] = (pos["SK_DPD"] > 0).astype(int)

    agg_dict = {
        "MONTHS_BALANCE": ["count", "min"],
        "CNT_INSTALMENT": ["mean", "max"],
        "CNT_INSTALMENT_FUTURE": ["mean", "min"],
        "SK_DPD": ["mean", "max"],               # số ngày trễ hạn (days past due)
        "SK_DPD_DEF": ["mean", "max"],
        "LATE_PAYMENT_FLAG": ["sum", "mean"],
    }
    return aggregate_table(pos, ID_COL, agg_dict, prefix="POS")


# ============================================================
# 5. CREDIT_CARD_BALANCE
# ============================================================

def build_credit_card_features(path):
    cc = pd.read_csv(path)
    cc["UTILIZATION_RATIO"] = safe_ratio(cc["AMT_BALANCE"], cc["AMT_CREDIT_LIMIT_ACTUAL"])
    cc["LATE_PAYMENT_FLAG"] = (cc["SK_DPD"] > 0).astype(int)

    agg_dict = {
        "MONTHS_BALANCE": ["count"],
        "AMT_BALANCE": ["mean", "max"],
        "AMT_CREDIT_LIMIT_ACTUAL": ["mean", "max"],
        "UTILIZATION_RATIO": ["mean", "max"],    # % hạn mức đã dùng — tín hiệu rủi ro rất mạnh
        "AMT_DRAWINGS_CURRENT": ["mean", "sum"],
        "AMT_PAYMENT_CURRENT": ["mean", "sum"],
        "SK_DPD": ["mean", "max"],
        "LATE_PAYMENT_FLAG": ["sum", "mean"],
    }
    return aggregate_table(cc, ID_COL, agg_dict, prefix="CC")


# ============================================================
# 6. INSTALLMENTS_PAYMENTS
# ============================================================

def build_installments_features(path):
    ins = pd.read_csv(path)
    ins["PAYMENT_DIFF"] = ins["AMT_INSTALMENT"] - ins["AMT_PAYMENT"]
    ins["PAYMENT_RATIO"] = safe_ratio(ins["AMT_PAYMENT"], ins["AMT_INSTALMENT"])
    ins["DAYS_LATE"] = ins["DAYS_ENTRY_PAYMENT"] - ins["DAYS_INSTALMENT"]
    ins["LATE_FLAG"] = (ins["DAYS_LATE"] > 0).astype(int)

    agg_dict = {
        "NUM_INSTALMENT_VERSION": ["nunique"],   # số lần hợp đồng trả góp bị thay đổi kỳ hạn
        "PAYMENT_DIFF": ["mean", "sum"],         # trả thiếu (dương) hay đủ/dư (âm)
        "PAYMENT_RATIO": ["mean", "min"],
        "DAYS_LATE": ["mean", "max"],            # trễ hạn bao nhiêu ngày
        "LATE_FLAG": ["sum", "mean"],
        "AMT_INSTALMENT": ["mean", "sum"],
        "AMT_PAYMENT": ["mean", "sum"],
    }
    return aggregate_table(ins, ID_COL, agg_dict, prefix="INSTALL")


# ============================================================
# 7. TỔNG HỢP TẤT CẢ MODALITY
# ============================================================

def run_feature_engineering():
    print("Loading application (core)...")
    app_df, app_cols, y = build_application_features(PATHS["application_train"])
    base_ids = app_df[ID_COL].values

    X_application = app_df[app_cols].fillna(0).values.astype(np.float32)
    mask_application_feat = (~app_df[app_cols].isna()).astype(np.float32).values
    print(f"  -> {X_application.shape[0]} khách hàng, {X_application.shape[1]} feature")

    modal_data = {
        "application": {"X": X_application, "mask_feat": mask_application_feat, "cols": app_cols}
    }
    # application luôn có -> modality mask toàn 1
    modality_masks = [np.ones((len(base_ids), 1), dtype=np.float32)]
    modality_names = ["application"]

    optional_builders = {
        "bureau": lambda: build_bureau_features(PATHS["bureau"], PATHS["bureau_balance"]),
        "previous": lambda: build_previous_features(PATHS["previous_application"]),
        "pos": lambda: build_pos_features(PATHS["pos_cash"]),
        "credit_card": lambda: build_credit_card_features(PATHS["credit_card"]),
        "installments": lambda: build_installments_features(PATHS["installments"]),
    }

    for name, builder in optional_builders.items():
        print(f"Building modality: {name} ...")
        agg_df = builder()
        X, feat_mask, mod_mask, cols = make_modality_matrix(base_ids, agg_df)

        modal_data[name] = {"X": X, "mask_feat": feat_mask, "cols": cols}
        modality_masks.append(mod_mask)
        modality_names.append(name)
        print(f"  -> shape={X.shape}, availability={mod_mask.mean():.2%}")

    level2_modality_mask = np.hstack(modality_masks).astype(np.float32)

    print("\n===== SUMMARY =====")
    for i, name in enumerate(modality_names):
        shape = modal_data[name]["X"].shape
        print(f"{name:<15}: shape={shape}, availability={level2_modality_mask[:, i].mean():.2%}")
    if y is not None:
        pos_rate = y.mean()
        print(f"\nTARGET distribution: default={int(y.sum())} ({pos_rate:.2%}), total={len(y)}")

    return {
        "base_ids": base_ids,
        "y": y,
        "modal_data": modal_data,
        "modality_names": modality_names,
        "level2_modality_mask": level2_modality_mask,
    }


def save_outputs(result, out_dir):
    """Lưu ra .npz (dùng cho training) + .json metadata (dùng cho model & web app sau này)."""
    save_dict = {
        "base_ids": result["base_ids"],
        "y": result["y"] if result["y"] is not None else np.array([]),
        "level2_modality_mask": result["level2_modality_mask"],
    }
    for name, d in result["modal_data"].items():
        save_dict[f"X_{name}"] = d["X"]
        save_dict[f"mask_{name}_feat"] = d["mask_feat"]

    npz_path = os.path.join(out_dir, "home_credit_features.npz")
    np.savez_compressed(npz_path, **save_dict)

    meta = {
        "modality_names": result["modality_names"],
        "feature_names": {name: d["cols"] for name, d in result["modal_data"].items()},
        "id_col": ID_COL,
        "label_col": LABEL_COL,
    }
    meta_path = os.path.join(out_dir, "home_credit_feature_metadata.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print(f"\nSaved: {npz_path}")
    print(f"Saved: {meta_path}")
    return npz_path, meta_path


if __name__ == "__main__":
    result = run_feature_engineering()
    save_outputs(result, PATHS["output_dir"])

Loading application (core)...
  -> 307511 khách hàng, 257 feature
Building modality: bureau ...
  -> shape=(307511, 19), availability=85.69%
Building modality: previous ...
  -> shape=(307511, 16), availability=94.65%
Building modality: pos ...
  -> shape=(307511, 12), availability=94.12%
Building modality: credit_card ...
  -> shape=(307511, 15), availability=28.26%
Building modality: installments ...
  -> shape=(307511, 13), availability=94.84%

===== SUMMARY =====
application    : shape=(307511, 257), availability=100.00%
bureau         : shape=(307511, 19), availability=85.69%
previous       : shape=(307511, 16), availability=94.65%
pos            : shape=(307511, 12), availability=94.12%
credit_card    : shape=(307511, 15), availability=28.26%
installments   : shape=(307511, 13), availability=94.84%

TARGET distribution: default=24825 (8.07%), total=307511

Saved: ./outputs\home_credit_features.npz
Saved: ./outputs\home_credit_feature_metadata.json


In [7]:
import numpy as np

# Load file lên
data = np.load(r"D:\2026\home_credit\outputs\home_credit_features.npz")

# Liệt kê tất cả các "chìa khóa" (tên ma trận) có trong file
print("Các mảng có trong file:", data.files)

# Xem kích thước của một số ma trận
print("Kích thước X_application:", data['X_application'].shape)
print("Kích thước nhãn y:", data['y'].shape)
print("Kích thước Modality Mask:", data['level2_modality_mask'].shape)

# Xem thử 5 dòng đầu của Modality Mask
print("5 dòng đầu của Modality Mask:\n", data['level2_modality_mask'][:5])

Các mảng có trong file: ['base_ids', 'y', 'level2_modality_mask', 'X_application', 'mask_application_feat', 'X_bureau', 'mask_bureau_feat', 'X_previous', 'mask_previous_feat', 'X_pos', 'mask_pos_feat', 'X_credit_card', 'mask_credit_card_feat', 'X_installments', 'mask_installments_feat']
Kích thước X_application: (307511, 257)
Kích thước nhãn y: (307511,)
Kích thước Modality Mask: (307511, 6)
5 dòng đầu của Modality Mask:
 [[1. 1. 1. 1. 0. 1.]
 [1. 1. 1. 1. 0. 1.]
 [1. 1. 1. 1. 0. 1.]
 [1. 0. 1. 1. 1. 1.]
 [1. 1. 1. 1. 0. 1.]]


In [9]:
import json

# Đường dẫn tới file metadata bạn đã tạo
meta_path = r"D:\2026\home_credit\outputs\home_credit_feature_metadata.json"

with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

print(f"{'EXPERT (MODALITY)':<25} | {'SỐ LƯỢNG FEATURE':<15}")
print("-" * 45)

for expert_name, features in meta["feature_names"].items():
    print(f"{expert_name.upper():<25} | {len(features):<15}")
    
    # In ra 10 feature đầu tiên của mỗi expert để kiểm tra
    print(f"   Samples: {', '.join(features[:8])}...")
    print("-" * 45)

# Nếu bạn muốn xem toàn bộ danh sách feature của một Expert cụ thể (ví dụ: Installments)
expert_to_check = "installments"
if expert_to_check in meta["feature_names"]:
    print(f"\nCHI TIẾT FEATURE CỦA EXPERT '{expert_to_check.upper()}':")
    for i, col in enumerate(meta["feature_names"][expert_to_check]):
        print(f"{i+1}. {col}")

EXPERT (MODALITY)         | SỐ LƯỢNG FEATURE
---------------------------------------------
APPLICATION               | 257            
   Samples: NAME_CONTRACT_TYPE, CODE_GENDER, FLAG_OWN_CAR, FLAG_OWN_REALTY, CNT_CHILDREN, AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY...
---------------------------------------------
BUREAU                    | 19             
   Samples: BUREAU_SK_ID_BUREAU_COUNT, BUREAU_CREDIT_ACTIVE_FLAG_SUM, BUREAU_CREDIT_ACTIVE_FLAG_MEAN, BUREAU_DAYS_CREDIT_MIN, BUREAU_DAYS_CREDIT_MAX, BUREAU_DAYS_CREDIT_MEAN, BUREAU_CREDIT_DAY_OVERDUE_MAX, BUREAU_CREDIT_DAY_OVERDUE_MEAN...
---------------------------------------------
PREVIOUS                  | 16             
   Samples: PREV_SK_ID_PREV_COUNT, PREV_AMT_ANNUITY_MEAN, PREV_AMT_ANNUITY_MAX, PREV_AMT_APPLICATION_MEAN, PREV_AMT_APPLICATION_MAX, PREV_AMT_CREDIT_MEAN, PREV_AMT_CREDIT_MAX, PREV_APP_CREDIT_RATIO_MEAN...
---------------------------------------------
POS                       | 12             
   Samples: PO

In [10]:
import numpy as np
import json
import pandas as pd

# 1. Load dữ liệu
data = np.load("./outputs/home_credit_features.npz")
with open("./outputs/home_credit_feature_metadata.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

experts = ["bureau", "previous", "pos", "credit_card", "installments"]
total_customers = data['base_ids'].shape[0]

print(f"Tổng số khách hàng trong bộ dữ liệu: {total_customers}\n")

# 2. Duyệt qua từng Expert và tính toán
for expert in experts:
    mask_key = f"mask_{expert}_feat"
    
    if mask_key in data:
        mask_array = data[mask_key]  # Ma trận (n_customers, n_features)
        feat_names = meta["feature_names"][expert]
        
        # Tính số lượng có dữ liệu (sum theo cột)
        counts = np.sum(mask_array, axis=0)
        availability_pct = (counts / total_customers) * 100
        missing_pct = 100 - availability_pct
        
        # Tạo bảng thống kê cho expert này
        df_stat = pd.DataFrame({
            'Feature Name': feat_names,
            'Available Count': counts.astype(int),
            'Availability (%)': availability_pct,
            'Missing (%)': missing_pct
        })
        
        print(f"=== THỐNG KÊ EXPERT: {expert.upper()} ===")
        # In ra 5 feature thiếu nhiều nhất và 5 feature đầy đủ nhất của Expert này
        print("-- 5 Feature thiếu nhiều nhất:")
        print(df_stat.sort_values('Missing (%)', ascending=False).head(5).to_string(index=False))
        
        # Tính trung bình cho cả Expert
        avg_missing = df_stat['Missing (%)'].mean()
        print(f"\n>> Tỷ lệ thiếu trung bình của Expert {expert.upper()}: {avg_missing:.2f}%")
        print("-" * 60 + "\n")
    else:
        print(f"Expert {expert} không tìm thấy trong file dữ liệu.\n")

Tổng số khách hàng trong bộ dữ liệu: 307511

=== THỐNG KÊ EXPERT: BUREAU ===
-- 5 Feature thiếu nhiều nhất:
                    Feature Name  Available Count  Availability (%)  Missing (%)
BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN           242442         78.840103    21.159897
 BUREAU_AMT_CREDIT_SUM_DEBT_MEAN           256131         83.291656    16.708344
      BUREAU_AMT_CREDIT_SUM_MEAN           263490         85.684738    14.315262
       BUREAU_AMT_CREDIT_SUM_MAX           263490         85.684738    14.315262
  BUREAU_CREDIT_ACTIVE_FLAG_MEAN           263491         85.685059    14.314941

>> Tỷ lệ thiếu trung bình của Expert BUREAU: 14.80%
------------------------------------------------------------

=== THỐNG KÊ EXPERT: PREVIOUS ===
-- 5 Feature thiếu nhiều nhất:
              Feature Name  Available Count  Availability (%)  Missing (%)
     PREV_AMT_ANNUITY_MEAN           290640         94.513687     5.486313
      PREV_AMT_ANNUITY_MAX           290640         94.513687     5.486313
 